# CHSH / Bell Inequality Exercise, Solutions

In [1]:
!pip install qiskit --quiet
!pip install qiskit[visualization] --quiet
!pip install qiskit_aer --quiet


## Answer

In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer
import numpy as np

backend = Aer.get_backend('aer_simulator')

def correlator(angle_a, angle_b, shots=20000):
    qc = QuantumCircuit(2, 2)

    # SOLUTION:
    qc.h(0)
    qc.cx(0, 1)
    qc.ry(-angle_a, 0)
    qc.ry(-angle_b, 1)
    qc.measure([0, 1], [0, 1])

    job = backend.run(qc, shots=shots)
    counts = job.result().get_counts()

    total = sum(counts.values())
    same = counts.get('00', 0) + counts.get('11', 0)
    diff = counts.get('01', 0) + counts.get('10', 0)
    E = (same - diff) / total

    return E


In [3]:
a, a_prime = 0, np.pi/2
b, b_prime = np.pi/4, 3*np.pi/4

E_ab   = correlator(a, b)
E_abp  = correlator(a, b_prime)
E_apb  = correlator(a_prime, b)
E_apbp = correlator(a_prime, b_prime)

S = E_ab - E_abp + E_apb + E_apbp

print("E(a,b)   =", round(E_ab, 4))
print("E(a,b')  =", round(E_abp, 4))
print("E(a',b)  =", round(E_apb, 4))
print("E(a',b') =", round(E_apbp, 4))
print("S =", round(S, 4))
print("Classical bound: 2   Quantum (Tsirelson) bound:", round(2*np.sqrt(2), 4))


E(a,b)   = 0.7109
E(a,b')  = -0.7034
E(a',b)  = 0.7033
E(a',b') = 0.7
S = 2.8176
Classical bound: 2   Quantum (Tsirelson) bound: 2.8284


**Expected result:** S should come out close to $2.828$ ($2\sqrt{2}$), clearly above the classical bound of 2. With 20000 shots per correlator you should land within about 0.02 of the theoretical value, since this is genuine sampling noise, not an error.

This confirms the entangled qubits produce correlations no local hidden variable theory can reproduce.


## Extension: unentangled state

In [4]:
def correlator_unentangled(angle_a, angle_b, shots=20000):
    qc = QuantumCircuit(2, 2)
    # no h, no cx: qubits stay separable, both start at |0>
    qc.ry(-angle_a, 0)
    qc.ry(-angle_b, 1)
    qc.measure([0, 1], [0, 1])

    counts = backend.run(qc, shots=shots).result().get_counts()
    total = sum(counts.values())
    same = counts.get('00', 0) + counts.get('11', 0)
    diff = counts.get('01', 0) + counts.get('10', 0)
    return (same - diff) / total

E_ab2   = correlator_unentangled(a, b)
E_abp2  = correlator_unentangled(a, b_prime)
E_apb2  = correlator_unentangled(a_prime, b)
E_apbp2 = correlator_unentangled(a_prime, b_prime)

S2 = E_ab2 - E_abp2 + E_apb2 + E_apbp2
print("S (unentangled) =", round(S2, 4))


S (unentangled) = 1.4039


**Expected result:** S for the unentangled $|00\rangle$ state comes out around $1.41$ ($\sqrt{2}$), well under both the quantum bound and the classical bound of 2. Without entanglement, there is no violation at all: the correlations are exactly what you'd get from two independent qubits, each responding only to its own measurement setting.
